# JPY Regime Dependency Phase 1
計画SHA: 11541f39730fcbae7dd7dfb5fc82c95bf7c18652。固定Baselineを読み込む診断研究。live変更なし。2022–2026は既閲覧。
実行前にBaselineと監査済みUSDJPY M1の8本を/contentへ用意するか、既存Driveを入力としてマウントしてください。

## 検証済み結果（2026-09-13）
Primary・Robustnessとも円安レジーム依存を支持せず。

| Long群 | 円安AvgR | 円高AvgR | pooled差 | 等重み差 | 正差戦略 |
|---|---:|---:|---:|---:|---:|
| 126日 | 0.094361 | 0.098400 | -0.004040 | -0.026011 | 5/10 |
| 200MA | 0.075584 | 0.119854 | -0.044271 | -0.043047 | 2/9 |

Baseline 16,298件のhash一致。Long 6,333件。履歴不足: Primary217件、Robustness386件。16_UJ_T10AはRobustnessの主要判定からLOW_SAMPLE除外。Phase 2へ進む条件は未充足。以下のセルで固定仕様を再実行し、10戦略の全指標を表示します。

In [ ]:
from pathlib import Path
import urllib.request, sys
IMPLEMENTATION_SHA = "a57735bf817281bc953107a2e05edd616367686a"
root = Path("/content/jpy_regime_phase1_code"); root.mkdir(exist_ok=True)
for path in ["src/research/jpy_regime_phase1.py", "src/research/edge_decay_analysis.py", "src/research/daily_stop_baseline_revalidation.py", "tests/verify_jpy_regime_phase1.py", "tests/test_jpy_regime_phase1.py"]:
    url = f"https://raw.githubusercontent.com/TR-KJ/time-entry-portfolio-lab/{IMPLEMENTATION_SHA}/{path}"
    (root / Path(path).name).write_bytes(urllib.request.urlopen(url).read())
sys.path.insert(0, str(root))


In [ ]:
# Drive入力が必要な場合のみTrue。成果物のDrive保存は末尾の別セル。
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
BASELINE = "/content/daily_stop_baseline_trades.csv"
M1_ROOT = "/content" # 必要なら監査済み8ファイルを含むDriveフォルダ


In [ ]:
import unittest, test_jpy_regime_phase1
result = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromModule(test_jpy_regime_phase1))
assert result.wasSuccessful()
from jpy_regime_phase1 import run
tables = run(BASELINE, M1_ROOT, "/content", IMPLEMENTATION_SHA)
from verify_jpy_regime_phase1 import verify
verify("/content", BASELINE)


In [ ]:
import pandas as pd
from IPython.display import display
for name in ["decision", "group_summary", "strategy_primary", "strategy_robustness", "coverage"]:
    print(name)
    display(pd.DataFrame(tables[name]))


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/time-entry-portfolio-lab/jpy_regime_phase1")
    destination.mkdir(parents=True, exist_ok=True)
    for p in Path("/content").glob("jpy_regime_phase1_*.csv"):
        shutil.copy2(p, destination / p.name)
